# InSight fitted daily solar energy from E-0771

This notebook estimates full-sol solar-array energy using only `E-0771` current and its matching voltage channel `E-0769`.

Workflow:

1. Read local InSight engineering `.tab` files.
2. Treat `9999.9`/`9999.900`/`9999.999` as missing values.
3. For each sol, fit the `E-0771` current curve with a cosine plus Gaussian form:

`I(t) = A*cos(w*(t-t0)) - B*exp(-(t-tc)**2/(2*sigma**2))`

4. Clip the fitted current at zero for physical power integration.
5. For each sol, compute one fixed voltage as the daily mean of valid `E-0769` samples.
6. Estimate hourly and daily energy from `Power_W(t) = daily_mean_E0769 * fitted_E0771(t)`.


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit

try:
    from tqdm.auto import tqdm
except ImportError:
    tqdm = lambda iterable, **kwargs: iterable


project_dir = next(
    p for p in [Path.cwd(), *Path.cwd().parents]
    if (p / "insight_engineering_data").exists()
)
data_dir = project_dir / "insight_engineering_data"

output_hourly_csv = project_dir / "InSight_hourly_solar_energy_filled.csv"
output_daily_csv = project_dir / "InSight_daily_solar_energy_filled.csv"
output_fit_params_csv = project_dir / "InSight_daily_E0771_fit_parameters.csv"
output_daily_fig = project_dir / "InSight_daily_solar_energy_filled.png"
output_hourly_fig = project_dir / "InSight_mean_hourly_solar_power_filled.png"
output_sol04_fit_fig = project_dir / "SOL0004_E0771_cosine_gaussian_fit.png"
output_sol400_fit_fig = project_dir / "SOL0400_E0771_cosine_gaussian_fit.png"

voltage_channel = "E-0769"
current_channel = "E-0771"
missing_threshold = 9999
required_cols = ["SCET_UTC", "SOL/Mars local time", voltage_channel, current_channel]

# One Mars local hour is 1/24 of a sol, expressed in Earth hours for Wh integration.
mars_local_hour_earth_h = 88775.244 / 3600 / 24
integration_points_per_hour = 61
minimum_fit_samples = 8
plot_target_sols = [4, 400]

tab_files = sorted(data_dir.rglob("*.tab"))

print("data_dir:", data_dir)
print("tab files:", len(tab_files))
print("voltage channel:", voltage_channel)
print("current channel:", current_channel)
print("Mars local hour in Earth hours:", mars_local_hour_earth_h)


In [ ]:
def e0771_model(t, A, w, t0, B, tc, sigma):
    return A * np.cos(w * (t - t0)) - B * np.exp(-((t - tc) ** 2) / (2 * sigma ** 2))


def read_tab(path):
    df = pd.read_csv(path, sep=",", skipinitialspace=True, low_memory=False)
    df.columns = df.columns.str.strip()
    missing = [c for c in required_cols if c not in df.columns]
    if missing:
        raise ValueError(f"missing columns: {missing}")
    return df[required_cols].copy()


def parse_mlst(sol_time):
    parts = sol_time.astype(str).str.extract(
        r"(?P<SOL>SOL-\d+)M(?P<hour>\d+):(?P<minute>\d+):(?P<second>\d+(?:\.\d+)?)"
    )
    local_hour = (
        parts["hour"].astype(float)
        + parts["minute"].astype(float) / 60
        + parts["second"].astype(float) / 3600
    )
    return parts["SOL"], parts["hour"].astype(float), local_hour


def file_to_samples(path):
    df = read_tab(path)
    df[voltage_channel] = pd.to_numeric(df[voltage_channel], errors="coerce")
    df[current_channel] = pd.to_numeric(df[current_channel], errors="coerce")
    df.loc[df[voltage_channel] >= missing_threshold, voltage_channel] = np.nan
    df.loc[df[current_channel] >= missing_threshold, current_channel] = np.nan

    df["SCET_UTC"] = pd.to_datetime(
        df["SCET_UTC"].astype(str).str.strip(),
        format="%Y-%jT%H:%M:%S.%f",
        errors="coerce",
    )
    df["SOL"], df["MLST_hour"], df["local_hour"] = parse_mlst(df["SOL/Mars local time"])
    df["SOL_number"] = df["SOL"].str.extract(r"SOL-(\d+)").astype(float)
    return df.dropna(subset=["SOL", "SOL_number", "local_hour"])


def fit_one_sol(group):
    sol = group["SOL"].iloc[0]
    sol_number = float(group["SOL_number"].iloc[0])
    voltage = group[voltage_channel]
    current = group[current_channel]
    voltage_mean = voltage.dropna().mean()

    valid = group[["local_hour", current_channel]].dropna().sort_values("local_hour")
    x = valid["local_hour"].to_numpy(dtype=float)
    y = valid[current_channel].to_numpy(dtype=float)

    result = {
        "SOL": sol,
        "SOL_number": sol_number,
        "Voltage_mean_V": voltage_mean,
        "Voltage_valid_samples": int(voltage.notna().sum()),
        "Current_valid_samples": int(current.notna().sum()),
        "A": np.nan,
        "w": np.nan,
        "t0": np.nan,
        "B": np.nan,
        "tc": np.nan,
        "sigma": np.nan,
        "fit_success": False,
        "fit_rmse_A": np.nan,
        "Peak_fitted_current_A": np.nan,
    }

    if len(x) < minimum_fit_samples or not np.isfinite(voltage_mean):
        return result

    peak_idx = int(np.nanargmax(y))
    peak_hour = float(x[peak_idx])
    peak_current = float(max(y[peak_idx], 0.01))
    p0 = [peak_current / 2, np.pi / 12, peak_hour, -peak_current / 2, peak_hour, 2.5]
    bounds = (
        [0.0, 0.12, 0.0, -2.0, 0.0, 0.2],
        [2.0, 0.50, 24.0, 2.0, 24.0, 8.0],
    )

    try:
        params, _ = curve_fit(
            e0771_model,
            x,
            y,
            p0=p0,
            bounds=bounds,
            maxfev=30000,
        )
        fitted = e0771_model(x, *params)
        rmse = float(np.sqrt(np.mean((fitted - y) ** 2)))
        dense_t = np.linspace(0, 24, 24 * integration_points_per_hour)
        dense_i = np.clip(e0771_model(dense_t, *params), 0, None)

        result.update(
            {
                "A": params[0],
                "w": params[1],
                "t0": params[2],
                "B": params[3],
                "tc": params[4],
                "sigma": params[5],
                "fit_success": True,
                "fit_rmse_A": rmse,
                "Peak_fitted_current_A": float(np.nanmax(dense_i)),
            }
        )
    except Exception as exc:
        result["fit_error"] = str(exc)

    return result


def integrate_fit_by_hour(row):
    if not row["fit_success"] or not np.isfinite(row["Voltage_mean_V"]):
        return []

    params = [row["A"], row["w"], row["t0"], row["B"], row["tc"], row["sigma"]]
    rows = []
    for hour in range(24):
        t = np.linspace(hour, hour + 1, integration_points_per_hour)
        current = np.clip(e0771_model(t, *params), 0, None)
        mean_current = np.trapezoid(current, t)
        mean_power = row["Voltage_mean_V"] * mean_current
        energy = mean_power * mars_local_hour_earth_h
        rows.append(
            {
                "SOL": row["SOL"],
                "SOL_number": row["SOL_number"],
                "MLST_hour": float(hour),
                "Voltage_mean_V": row["Voltage_mean_V"],
                "Fitted_mean_current_A": mean_current,
                "Fitted_mean_power_W": mean_power,
                "Filled_hour_energy_Wh": energy,
                "fit_success": row["fit_success"],
                "fit_rmse_A": row["fit_rmse_A"],
            }
        )
    return rows


In [ ]:
sample_parts = []
errors = []

for path in tqdm(tab_files):
    try:
        samples = file_to_samples(path)
        if not samples.empty:
            sample_parts.append(samples)
    except Exception as exc:
        errors.append((str(path), str(exc)))

print("processed files:", len(sample_parts))
print("errors:", len(errors))
if errors[:5]:
    print(errors[:5])
if not sample_parts:
    raise RuntimeError("No telemetry samples were processed.")

samples = pd.concat(sample_parts, ignore_index=True)
samples = samples.sort_values(["SOL_number", "local_hour", "SCET_UTC"])
print("samples:", len(samples))
display(samples[["SOL", "SOL_number", "local_hour", voltage_channel, current_channel]].head())


In [ ]:
fit_rows = []

for _, group in tqdm(samples.groupby(["SOL", "SOL_number"], sort=True), total=samples[["SOL", "SOL_number"]].drop_duplicates().shape[0]):
    fit_rows.append(fit_one_sol(group))

fit_params = pd.DataFrame(fit_rows).sort_values("SOL_number")
print("sols fitted:", len(fit_params))
print("fit successes:", int(fit_params["fit_success"].sum()))
print("fit failures:", int((~fit_params["fit_success"]).sum()))
display(fit_params.head())


In [ ]:
hourly_rows = []
for _, row in fit_params.iterrows():
    hourly_rows.extend(integrate_fit_by_hour(row))

hourly = pd.DataFrame(hourly_rows)
if hourly.empty:
    raise RuntimeError("No hourly energy rows were produced from fitted current curves.")

hourly = hourly.sort_values(["SOL_number", "MLST_hour"])
daily = (
    hourly.groupby(["SOL", "SOL_number"], as_index=False)
    .agg(
        Filled_daily_energy_Wh=("Filled_hour_energy_Wh", "sum"),
        Voltage_mean_V=("Voltage_mean_V", "first"),
        Peak_fitted_current_A=("Fitted_mean_current_A", "max"),
        Mean_fitted_power_W=("Fitted_mean_power_W", "mean"),
        fit_rmse_A=("fit_rmse_A", "first"),
    )
    .merge(
        fit_params[
            [
                "SOL",
                "SOL_number",
                "Voltage_valid_samples",
                "Current_valid_samples",
                "fit_success",
                "A",
                "w",
                "t0",
                "B",
                "tc",
                "sigma",
            ]
        ],
        on=["SOL", "SOL_number"],
        how="left",
    )
    .sort_values("SOL_number")
)

hourly.to_csv(output_hourly_csv, index=False, encoding="utf-8-sig")
daily.to_csv(output_daily_csv, index=False, encoding="utf-8-sig")
fit_params.to_csv(output_fit_params_csv, index=False, encoding="utf-8-sig")

print("saved hourly csv:", output_hourly_csv)
print("saved daily csv:", output_daily_csv)
print("saved fit parameters csv:", output_fit_params_csv)
print("daily fitted Wh/sol min mean median max:")
print(daily["Filled_daily_energy_Wh"].agg(["min", "mean", "median", "max"]))
display(daily.head())


In [ ]:
def plot_fit_for_sol(sol_number, output_path):
    target = fit_params.loc[fit_params["SOL_number"].eq(float(sol_number))]
    if target.empty:
        raise RuntimeError(f"No fit row found for SOL-{sol_number:04d}")
    row = target.iloc[0]
    if not row["fit_success"]:
        raise RuntimeError(f"Fit failed for SOL-{sol_number:04d}")

    obs = samples.loc[samples["SOL_number"].eq(float(sol_number))].dropna(subset=[current_channel])
    params = [row["A"], row["w"], row["t0"], row["B"], row["tc"], row["sigma"]]
    t = np.linspace(0, 24, 24 * integration_points_per_hour)
    fitted = np.clip(e0771_model(t, *params), 0, None)

    plt.figure(figsize=(9, 4.8))
    plt.scatter(obs["local_hour"], obs[current_channel], s=10, alpha=0.45, label="observed E-0771")
    plt.plot(t, fitted, color="tab:red", lw=2.0, label="cosine - Gaussian fit")
    plt.xlabel("Mars local solar time hour")
    plt.ylabel("E-0771 current (A)")
    plt.title(f"SOL-{sol_number:04d} E-0771 cosine-Gaussian fit")
    plt.xlim(0, 24)
    plt.xticks(np.arange(0, 25, 2))
    plt.grid(alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.savefig(output_path, dpi=200)
    plt.show()
    print("saved fit figure:", output_path)


plot_fit_for_sol(4, output_sol04_fit_fig)
plot_fit_for_sol(400, output_sol400_fit_fig)


In [ ]:
sol_fit_range = (4, 25)
sol_range_fit_fig = Path("SOL0004_SOL0025_E0771_fitted_curves.png")

if "fit_params" not in globals():
    if output_fit_params_csv.exists():
        fit_params = pd.read_csv(output_fit_params_csv)
    else:
        raise RuntimeError("Run the fit-current cell first, or create output_fit_params_csv.")

plot_rows = (
    fit_params.loc[
        fit_params["SOL_number"].between(sol_fit_range[0], sol_fit_range[1])
        & fit_params["fit_success"].astype(bool)
    ]
    .sort_values("SOL_number")
    .copy()
)

if plot_rows.empty:
    raise RuntimeError(f"No successful fitted curves found for sols {sol_fit_range}.")

t = np.linspace(0, 24, 24 * integration_points_per_hour)
colors = plt.cm.viridis(np.linspace(0, 1, len(plot_rows)))

plt.figure(figsize=(10, 5.5))
for color, (_, row) in zip(colors, plot_rows.iterrows()):
    params = [row["A"], row["w"], row["t0"], row["B"], row["tc"], row["sigma"]]
    fitted_current = np.clip(e0771_model(t, *params), 0, None)
    plt.plot(t, fitted_current, lw=1.3, color=color, alpha=0.9, label=f"SOL-{int(row['SOL_number']):04d}")

plt.xlabel("Mars local solar time hour")
plt.ylabel("Fitted E-0771 current (A)")
plt.title("SOL-0004 to SOL-0025 fitted E-0771 current curves")
plt.xlim(0, 24)
plt.xticks(np.arange(0, 25, 2))
plt.grid(alpha=0.3)
plt.legend(ncol=2, fontsize=8, frameon=True)
plt.tight_layout()
plt.savefig(sol_range_fit_fig, dpi=200)
plt.show()

print("plotted sols:", int(plot_rows["SOL_number"].min()), "to", int(plot_rows["SOL_number"].max()))
print("saved sol-range fit figure:", sol_range_fit_fig.resolve())


In [ ]:
plt.figure(figsize=(13, 4.5))
plt.plot(
    daily["SOL_number"],
    daily["Filled_daily_energy_Wh"],
    color="tab:blue",
    lw=1.5,
    label="daily mean E-0769 * fitted E-0771",
)
plt.xlabel("Sol")
plt.ylabel("Estimated energy per sol (Wh)")
plt.title("InSight daily solar-array energy from fitted E-0771 current")
plt.grid(alpha=0.3)
plt.xlim(0,200)
plt.legend()
plt.tight_layout()
plt.savefig(output_daily_fig, dpi=200)
plt.show()

mean_hourly = hourly.groupby("MLST_hour", as_index=False).agg(
    fitted_mean_power_W=("Fitted_mean_power_W", "mean"),
    fitted_mean_current_A=("Fitted_mean_current_A", "mean"),
)

plt.figure(figsize=(8, 4))
plt.plot(mean_hourly["MLST_hour"], mean_hourly["fitted_mean_power_W"], marker="o")
plt.xlabel("Mars local solar time hour")
plt.ylabel("Mean fitted power (W)")
plt.title("Mean fitted E-0769 * E-0771 power by local hour")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(output_hourly_fig, dpi=200)
plt.show()

print("saved daily figure:", output_daily_fig)
print("saved hourly figure:", output_hourly_fig)


In [ ]:
mean_coefficients_csv = Path("E0771_mean_fit_coefficients.csv")
mean_coefficients_fig = Path("E0771_mean_fit_curve_from_daily_coefficients.png")

if "fit_params" not in globals():
    if output_fit_params_csv.exists():
        fit_params = pd.read_csv(output_fit_params_csv)
    else:
        raise RuntimeError("Run the fit-current cell first, or create output_fit_params_csv.")

coefficient_cols = ["A", "w", "t0", "B", "tc", "sigma"]
coefficient_rows = fit_params.loc[
    fit_params["fit_success"].astype(bool),
    ["SOL", "SOL_number", *coefficient_cols, "fit_rmse_A", "Peak_fitted_current_A"],
].copy()
coefficient_rows = coefficient_rows.replace([np.inf, -np.inf], np.nan).dropna(subset=coefficient_cols)

if coefficient_rows.empty:
    raise RuntimeError("No successful finite E-0771 fit coefficients are available.")

mean_coefficients = coefficient_rows[coefficient_cols].mean()
coefficient_summary = pd.DataFrame(
    {
        "parameter": coefficient_cols,
        "mean": mean_coefficients.reindex(coefficient_cols).values,
        "median": coefficient_rows[coefficient_cols].median().reindex(coefficient_cols).values,
        "std": coefficient_rows[coefficient_cols].std().reindex(coefficient_cols).values,
    }
)
coefficient_summary.insert(0, "successful_fit_sols_used", len(coefficient_rows))
coefficient_summary.to_csv(mean_coefficients_csv, index=False, encoding="utf-8-sig")

t = np.linspace(0, 24, 24 * integration_points_per_hour)
mean_param_current = np.clip(e0771_model(t, *mean_coefficients.reindex(coefficient_cols).to_numpy()), 0, None)
daily_fit_matrix = np.vstack([
    np.clip(e0771_model(t, *row[coefficient_cols].to_numpy(dtype=float)), 0, None)
    for _, row in coefficient_rows.iterrows()
])
pointwise_mean_current = np.nanmean(daily_fit_matrix, axis=0)

plt.figure(figsize=(9.5, 5.0))
plt.plot(t, pointwise_mean_current, color="0.55", lw=1.5, ls="--", label="pointwise mean of daily fitted curves")
plt.plot(t, mean_param_current, color="tab:red", lw=2.4, label="curve from mean coefficients")
plt.xlabel("Mars local solar time hour")
plt.ylabel("Fitted E-0771 current (A)")
plt.title(f"Mean E-0771 fit coefficients across {len(coefficient_rows)} sols")
plt.xlim(0, 24)
plt.xticks(np.arange(0, 25, 2))
plt.grid(alpha=0.3)
plt.legend(frameon=True)
plt.tight_layout()
plt.savefig(mean_coefficients_fig, dpi=200)
plt.show()

print("successful fit sols used:", len(coefficient_rows))
print("saved mean coefficients csv:", mean_coefficients_csv.resolve())
print("saved mean coefficient curve figure:", mean_coefficients_fig.resolve())
try:
    display(coefficient_summary)
except NameError:
    print(coefficient_summary.to_string(index=False))


Notes:

- `B` is allowed to be positive or negative. The implemented functional form remains `A*cos(...) - B*Gaussian(...)`; a negative fitted `B` means the Gaussian term adds to the cosine.
- Fitted currents are clipped at zero before power integration.
- Daily voltage is fixed per sol as the mean of valid `E-0769` samples after removing `9999.9` sentinel values.
- Energy integration uses Mars local solar time: one local hour equals `88775.244 / 3600 / 24` Earth hours.
